# The InConvenience of Building RAG Without Langchain

1. Read the document
2. Split the document
    - May fail to generate a response due to token limit
    - Longer documents (longer input) take more time to generate responses
3. Embedding: Store in a vector database
4. When a question is asked, perform similarity search in the vector database
5. Pass the retrieved documents along with the question to the LLM

# 1. Read the document (문서의 내용을 읽는다)

In [ ]:
# langchain을 썼을 때는 document loader를 썼는데, 지금은 langchain을 안쓰니까 -> python-docx 이용
# %pip install python-docx

- docx is a ZIP-compressed XML file, so `open().read()` returns broken binary
- `python-docx` parses the internal XML and extracts text via `document.paragraphs`
- `paragraph.text` contains the full text of each paragraph (split by line breaks)
- All paragraphs are concatenated into `full_text` for later chunking

docx file -> parse with Document -> paragraph별 text 추출 -> full_text(usable)

In [ ]:
from docx import Document

document = Document("./tax.docx")
print(f"document == {document}")
# dir(document) : document가 가지고 있는 모든 속성과 method 이름 list 보여줌
print(f"document == {dir(document)}")

full_text = ""
for index, paragraph in enumerate(document.paragraphs):
    print(f"paragraph == {paragraph.text}")
    full_text += f"{paragraph.text}\n"

# 2. Split text (문서를 쪼갠다)

In [ ]:
# %pip install tiktoken

In [ ]:
# tiktoken : split text
import tiktoken


def split_text(full_text, chunk_size):
    encoder = tiktoken.encoding_for_model("gpt-4o")
    # encode = text -> 숫자 list
    total_encoding = encoder.encode(full_text)
    total_token_count = len(total_encoding)
    text_list = []

    for i in range(0, total_token_count, chunk_size):
        chunk = total_encoding[i : i + chunk_size]
        decoded = encoder.decode(chunk)
        text_list.append(decoded)

    return text_list

In [ ]:
chunk_list = split_text(full_text, 1500)

In [ ]:
chunk_list

# 3. Document Embedding

In [ ]:
# chroma가 제공하는 기본 기능이 있다. 그걸 이용해 embedding

# %pip install chromadb

In [ ]:
import chromadb

chroma_client = chromadb.Client()

In [ ]:
collection_name = "tax-collection"

In [ ]:
from chromadb.utils.embedding_functions import OllamaEmbeddingFunction

ollama_embedding = OllamaEmbeddingFunction(model_name="nomic-embed-text")

In [ ]:
tax_collection = chroma_client.get_or_create_collection(
    collection_name, embedding_function=ollama_embedding
)

In [ ]:
id_list = []
for index in range(len(chunk_list)):
    id_list.append(f"{index}")

In [ ]:
len(id_list)

In [ ]:
len(chunk_list)

In [ ]:
tax_collection.add(documents=chunk_list, ids=id_list)

# 4. Similarity Check (유사도 검색)

In [ ]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
retrieved_doc = tax_collection.query(query_texts=query)

In [ ]:
retrieved_doc["documents"][0]

# 5. LLM 질의

In [ ]:
import ollama

response = ollama.chat(
    model="llama3:latest",
    messages=[
        {
            "role": "system",
            "content": f'당신은 한국의 소득세 전문가 입니다. 아래 내용을 참고해서 사용자의 질문에 답변해주세요 {retrieved_doc["documents"][0]}',
        },
        {"role": "user", "content": query},
    ],
)

In [ ]:
response

In [ ]:
response.message.content